# PC-CDDM Kaggle 训练 Notebook

针对复杂电磁环境 (CEE) 下低 SNR IQ 信号去噪的条件扩散模型 — **训练 + 评估完整流程**。

## 工作流

1. **环境检查** — torch / GPU 信息
2. **拉取代码** — git clone + pip install
3. **路径自动检测** — 在 `/kaggle/input/` 下找数据集 H5 文件
4. **训练** — 会话感知, 默认 11 小时上限, 支持续训
5. **评估** — DDPM 完整反向链 (论文红线), 按 SNR 档位汇总
6. **打包结果** — `runs/` 目录打 zip, Save & Run All 提交后一次性下载

## 单会话 vs 多会话训练

- **单会话** (推荐, 配合心跳脚本): 默认 `paths.resume_from = null`, 一次跑完
- **多会话续训**: 上次结束后, 把 `runs/` 目录上传为 Private Dataset 挂载到本 Notebook,
  把数据复制到 `/kaggle/working/runs/` 后, 在第 4 步 override 改 `paths.resume_from = "auto"`

## 红线

- 评估必须 `use_ddim=False` 走完整 DDPM 反向链, DDIM 仅供 dev 调试
- 训练 loss 不是模型质量指标, 验证 NMSE 才是


## 1. 环境检查

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU      : {p.name}")
    print(f"VRAM     : {p.total_memory / 1e9:.1f} GB")
    print(f"Capability: sm_{p.major}{p.minor}")


## 2. 拉取代码

每次重新执行都会覆盖旧目录,确保拿到 GitHub 最新版本。

In [ ]:
import shutil
from pathlib import Path

REPO_URL = "https://github.com/luxixn/pc_cddm.git"
CODE_DIR = Path("/kaggle/working/pc_cddm")

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)

!git clone --depth 1 {REPO_URL} {CODE_DIR}


In [ ]:
%cd {CODE_DIR}
# Kaggle 已预装与 GPU 兼容的 torch — 重装会引发 cudaErrorNoKernelImageForDevice
# 这里只装 requirements.txt 里非 torch 的依赖
!pip install -q $(grep -vE '^(torch|#|$)' requirements.txt)


## 3. 路径自动检测

在 `/kaggle/input/` 下找 `.h5` 文件, 不写死 dataset slug。
若挂载了多个 dataset, 用下面的 `H5_PATH = ...` 手动指定。

In [ ]:
import glob
from pathlib import Path

candidates = sorted(glob.glob("/kaggle/input/**/*.h5", recursive=True))
print(f"找到 {len(candidates)} 个 H5 文件:")
for p in candidates:
    print(f"  {p}  ({Path(p).stat().st_size / 1e6:.1f} MB)")

assert len(candidates) > 0, "未在 /kaggle/input/ 下找到 .h5 文件, 请确认 dataset 已挂载"

# 多个文件时取第一个; 需要手动选请改这一行:
H5_PATH = candidates[0]
print(f"\n选用: {H5_PATH}")


## 4. 训练

调 `pc_cddm.train.main()` 跑训练循环。所有路径与 Kaggle 适配的参数通过 `overrides` 注入。

**续训怎么改**:

```python
"paths.resume_from": "auto",   # 自动找 ckpt_latest.pt
```

放在 overrides 字典里即可。前提是 `/kaggle/working/runs/kaggle_main/ckpts/ckpt_latest.pt` 存在
(从上次会话的 dataset 复制过来)。

In [ ]:
from pc_cddm.train import main as train_main

train_summary = train_main(
    "configs/default.yaml",
    overrides={
        # ---- Kaggle 路径注入 ----
        "paths.data":          H5_PATH,
        "paths.output_root":   "/kaggle/working/runs",
        "paths.exp_name":      "kaggle_main",

        # ---- 单会话留 1 小时 evaluate 余量 ----
        "train.max_wallclock_hours": 11.0,

        # ---- 续训打开下面这行 ----
        # "paths.resume_from": "auto",
    },
)

print()
print("=" * 60)
print("训练摘要:")
print("=" * 60)
for k, v in train_summary.items():
    print(f"  {k}: {v}")


## 5. 评估 (DDPM 完整反向链)

调 `pc_cddm.evaluate.main()`, 自动选 `ckpt_best.pt`, 跑 DDPM 全 1000 步 + PSD 重估。

输出位置: `/kaggle/working/runs/kaggle_main/eval/ddpm_full/`
- `eval_summary.yaml`  — 总体 + 各 SNR 档位
- `eval_per_sample.csv` — 逐样本指标
- `eval_log.txt` — 评估日志

In [ ]:
from pc_cddm.evaluate import main as eval_main

eval_summary = eval_main(
    "configs/default.yaml",
    ckpt_path=None,        # 自动取 ckpt_best.pt
    split="val",           # 论文表格用 val; 想跑全量改 "all"
    use_ddim=False,        # 红线: 论文必须 DDPM 全链
    save_per_sample=True,
    eval_name="ddpm_full",
    overrides={
        "paths.data":          H5_PATH,
        "paths.output_root":   "/kaggle/working/runs",
        "paths.exp_name":      "kaggle_main",
    },
)

print()
print("=" * 60)
print("评估总览:")
print("=" * 60)
o = eval_summary["overall"]
print(f"  N         = {o['n_samples']}")
print(f"  NMSE      = {o['nmse_mean']:.4f}")
print(f"  OutSNR dB = {o['out_snr_db_mean']:.2f}")
print(f"  InSNR  dB = {o['in_snr_db_mean']:.2f}")
print(f"  Gain   dB = {o['snr_gain_db_mean']:.2f}")
print()
print("按输入 SNR 档位:")
for snr, m in eval_summary["by_snr_db"].items():
    print(f"  SNR={snr:>5.1f} dB  N={m['n']:>4d}  "
          f"NMSE={m['nmse']:.4f}  OutSNR={m['out_snr_db']:>6.2f}dB  "
          f"Gain={m['snr_gain_db']:>5.2f}dB")


## 6. 打包结果

把 `runs/kaggle_main/` 打成 zip 放到 `/kaggle/working/`,
**Save & Run All** 提交后会出现在 Notebook Output 里, 一次下载就拿走全部 ckpt + 日志 + 评估报告。

In [ ]:
import shutil
from pathlib import Path

run_dir = Path("/kaggle/working/runs/kaggle_main")
zip_base = "/kaggle/working/run_results"   # 不带 .zip 后缀

shutil.make_archive(zip_base, "zip", run_dir)
zip_path = Path(zip_base + ".zip")
print(f"已打包: {zip_path}")
print(f"大小  : {zip_path.stat().st_size / 1e6:.1f} MB")
print()
print("run_dir 内文件:")
for p in sorted(run_dir.rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f"  {size_kb:>8.1f} KB  {p.relative_to(run_dir)}")
